# Lesson 7 — Correlation vs. Prediction vs. Causation

Read `README.md` (or `README.pl.md`) first. A coefficient in a linear model tells you three different things depending on what question you're asking — and they're easy to mix up.

In [ ]:
from task import impute_driver_experience, load_shipments, split_shipments

df = load_shipments()
train_df, test_df = split_shipments(df)
train_df, test_df = impute_driver_experience(train_df, test_df)

## Is num_stops's coefficient trustworthy?

Fit it alone, then fit it inside the full feature set. If the coefficient barely moves, it's at least stable — not proof of causation, but a good sign.

In [ ]:
from task import FEATURE_COLUMNS, coefficient_for, fit_model_on

model_alone = fit_model_on(train_df, ["num_stops"])
coef_alone = coefficient_for(model_alone, ["num_stops"], "num_stops")

model_full = fit_model_on(train_df, FEATURE_COLUMNS)
coef_full = coefficient_for(model_full, FEATURE_COLUMNS, "num_stops")

coef_alone, coef_full

## Is distance_km's coefficient trustworthy?

Same experiment, but add `planned_duration_min` — a column that's almost the same information as distance, twice.

In [ ]:
distance_alone = fit_model_on(train_df, ["distance_km"])
distance_coef_alone = coefficient_for(distance_alone, ["distance_km"], "distance_km")

distance_with_planned = fit_model_on(train_df, ["distance_km", "planned_duration_min"])
distance_coef_with_planned = coefficient_for(
    distance_with_planned, ["distance_km", "planned_duration_min"], "distance_km"
)

correlation = train_df["distance_km"].corr(train_df["planned_duration_min"])
distance_coef_alone, distance_coef_with_planned, correlation

## Correlation, prediction, and causation are three different questions

- **Correlation** (Lesson 3): does this column move together with delay? `weather` did, strongly — but it's categorical, so it never showed up in a numeric correlation matrix.
- **Prediction** (Lessons 5-6): does adding this column reduce error? `num_stops` did, and its coefficient stays put no matter what else is in the model.
- **Causation**: does *changing* this column change delay, all else equal? `distance_km`'s wildly unstable coefficient above means you cannot read a causal story off it alone — it's tangled up with `planned_duration_min`. `num_stops` is more trustworthy, but stability alone still isn't proof — it could still be a proxy for something else entirely (e.g. route complexity) that this dataset doesn't record.

## What can TransLine actually act on?

Weather: not actionable directly, but predictable — could be used to set expectations in advance. `num_stops`: possibly actionable (route planning), if it really is a cause and not just a proxy. `distance_km`: don't make a causal argument about it at all, given the instability above.

## Your notes

Pick one factor from this case (weather, num_stops, driver_experience_years, distance_km) and write two or three sentences: is it correlated, predictive, plausibly causal, actionable — and which of those are actually the same claim, and which are different?